In [ ]:
import os
import pandas as pd
import torch
import time

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)

print("\nGPU information:")

if torch.cuda.is_available():
    print("CUDA available : YES")
    print("Device         : GPU")
    print("GPU name       :", torch.cuda.get_device_name(0))
else:
    print("Device         : CPU")


## clmentbisaillon/fake-and-real-news-dataset (Option A)

### Preprocessing and Exploration

In [ ]:
data_path = "/kaggle/input/fake-and-real-news-dataset"

fake = pd.read_csv(f"{data_path}/Fake.csv")
real = pd.read_csv(f"{data_path}/True.csv")

print("Fake:", fake.shape)
print("Real:", real.shape)

In [ ]:
print(fake["subject"].value_counts())

In [ ]:
print(real["subject"].value_counts())

In [ ]:
fake["label"] = 0
real["label"] = 1

df = pd.concat([fake, real], ignore_index=True)
df.drop(columns=["date"], inplace=True)

print(df.shape)
print(df["label"].value_counts())

In [ ]:
pd.crosstab(df["subject"], df["label"])

In [ ]:
df.drop(columns=["subject"], inplace=True)

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = df["title"] + " " + df["text"]

df["content"] = (
    df["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[["content", "label"]]

In [ ]:
print("Before dropping duplicates:", df.shape)
df = df.drop_duplicates(subset=["content"])
print("After dropping duplicates:", df.shape)
print(df["label"].value_counts())

### Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=100000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.perf_counter()

model.fit(X_train_tfidf, y_train)

training_time = time.perf_counter() - start_time

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_test_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print(f"Accuracy for TF-IDF + Logistic Regression: {accuracy:.4f}")
print(f"F1 Score for TF-IDF + Logistic Regression: {f1:.4f}")
print(f"Training time for TF-IDF + Logistic Regression: {training_time:.2f} seconds")